# Tradepad
OVME-style valuation: pick an asset class and a product type (the library's instruments), fill the deal fields, pick a
model, price with greeks (same functions as the web app, `tqstudio/views/tradepad.py`).

## Product and deal

In [ ]:
ASSET_CLASS = "Equity"         # Equity | Rates
PRODUCT = "Vanilla"            # Equity: Vanilla, Digital, Barrier, Autocallable; Rates: Deposit, FRA, Swap, OIS,
                               #         Cap/Floor, Collar, Swaption
FIELDS = {"ticker": "TICKER1", "option_type": "Put", "strike_pct": 80.0, "expiry": "2027-11-05"}   # others: defaults
MODEL = None                   # None = the product's first model
N_PATHS = 50_000

In [ ]:
import os
import sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "marketdata").is_dir())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorquant as tq
from tqstudio.views.tradepad import ModelCache, catalog, price_trade

mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
cache = ModelCache()                                               # calibrated local vols, reused across pricings
spec = catalog(mkt)["asset_classes"][ASSET_CLASS][PRODUCT]
form = pd.DataFrame([{"field": f["name"], "label": f["label"], "value": FIELDS.get(f["name"], f["default"]),
                      "options": ", ".join(map(str, f.get("options", [])))} for f in spec["fields"]]).set_index("field")
print("models:", spec["models"])
form

## Valuation

In [ ]:
out = price_trade(mkt, cache, {"asset_class": ASSET_CLASS, "product": PRODUCT, "fields": FIELDS, "model": MODEL},
                  n_paths=N_PATHS)
rows = {("Valuation", label): value for label, value, _ in out["headline"]}
rows.update({("Details", label): value for label, value, _ in out["info"]})
print(f"{out['product']} · {out['model']}")
pd.Series(rows, dtype=object).to_frame("value")

## Vega and rho

In [ ]:
if out["vega"]:
    v = out["vega"]
    z = np.asarray(v["z"])
    scale = np.abs(z).max() or 1.0
    fig, ax = plt.subplots(figsize=(8, 3.8))
    im = ax.imshow(z, cmap="RdBu_r", vmin=-scale, vmax=scale, aspect="auto")
    for (i, j), x in np.ndenumerate(z):
        if abs(x) >= 0.01 * scale:
            ax.text(j, i, f"{x:.3g}", ha="center", va="center", fontsize=6.5)
    ax.set_xticks(range(len(v["x"])), v["x"], fontsize=7, rotation=45)
    ax.set_yticks(range(len(v["y"])), v["y"], fontsize=7)
    ax.set(xlabel=v["x_title"], ylabel=v["y_title"], title=v["title"])
    fig.colorbar(im, ax=ax)
rho = pd.DataFrame({r["curve"]: pd.Series(r["rho"], index=r["dates"]) for r in out["rho"]})
rho.T.round(2)

## Another product, same session

In [ ]:
swap = price_trade(mkt, cache, {"asset_class": "Rates", "product": "Swap",
                                "fields": {"ccy": "EUR", "tenor": "10Y", "rate": 2.8, "side": "Receiver"}})
pd.Series({label: value for label, value, _ in swap["headline"]}, dtype=object).to_frame("Swap 10Y receiver")